In [1]:
from langchain.schema.runnable import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
import dotenv
dotenv.load_dotenv()

True

### PromptTemplate 에 기사내용

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

prompt = PromptTemplate.from_template("""
다음의 context를 읽고, 질문에 답해줘.
context: {context}
질문: {question}
""")

chain = prompt | llm

response = chain.invoke({
    'context': '''
다음 기사를 근거로 질문에 답하세요.
-----
제21대 대한민국 대통령에 이재명 더불어민주당 후보가 당선됐다.

초유의 비상계엄 사태와 윤석열 전 대통령 파면 속에 치러진 조기 대선에서 민심이 정권 교체를 선택한 것이다. 전국 최종 투표율은 79.4%를 기록, 총 3524만416명이 투표에 참여했다.

중앙선거관리위원회에 따르면 4일 오전 5시 10분 개표율 100%를 기준으로 기호 1번 더불어민주당 이재명 후보가 1728만7513표로 전체 49.42%를 득표했다. 1439만5639표를 얻은 기호 2번 국민의힘 김문수 후보(41.15%)를 8.27%p차로 앞서며 당선을 확정 지었다.

앞서 방송 3사(KBS·MBC·SBS) 출구조사에서는 이재명 후보가 51.7%를 득표해 과반을 넘길 것이라 예측됐지만, 최종적으로 과반을 하지는 못했다.

이재명 후보는 제21대 대통령 당선이 확실시된 4일 오전 서울 여의도 국회 앞에 마련된 야외무대에 "민주공화국 대한민국 시민 여러분께 진심으로 감사드린다"며 "여러분들이 제게 기대하시고 맡긴 그 사명을 한순간도 잊지 않고 한 치의 어긋남도 없이 반드시, 확실히 이행하겠다"고 말했다.
-----''',
    'question': "한국의 대통령은?"
})
print(response.content)

이재명


### Hub에서 pull 한 prompt에 기사내용

In [5]:
from langchain import hub
prompt = hub.pull('rlm/rag-prompt')
print(prompt)

chain = prompt | llm

response = chain.invoke({
    'context': '''다음 기사를 근거로 질문에 답하세요.
-----
제21대 대한민국 대통령에 이재명 더불어민주당 후보가 당선됐다.
초유의 비상계엄 사태와 윤석열 전 대통령 파면 속에 치러진 조기 대선에서 민심이 정권 교체를 선택한 것이다. 전국 최종 투표율은 79.4%를 기록, 총 3524만416명이 투표에 참여했다.
중앙선거관리위원회에 따르면 4일 오전 5시 10분 개표율 100%를 기준으로 기호 1번 더불어민주당 이재명 후보가 1728만7513표로 전체 49.42%를 득표했다. 1439만5639표를 얻은 기호 2번 국민의힘 김문수 후보(41.15%)를 8.27%p차로 앞서며 당선을 확정 지었다.
앞서 방송 3사(KBS·MBC·SBS) 출구조사에서는 이재명 후보가 51.7%를 득표해 과반을 넘길 것이라 예측됐지만, 최종적으로 과반을 하지는 못했다.
이재명 후보는 제21대 대통령 당선이 확실시된 4일 오전 서울 여의도 국회 앞에 마련된 야외무대에 "민주공화국 대한민국 시민 여러분께 진심으로 감사드린다"며 "여러분들이 제게 기대하시고 맡긴 그 사명을 한순간도 잊지 않고 한 치의 어긋남도 없이 반드시, 확실히 이행하겠다"고 말했다.
-----''',
    'question': '한국의 대통령은?'
})
print(response.content)

input_variables=['context', 'question'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]
기사에 따르면 대한민국의 제21대 대통령은 이재명 더불어민주당 후보입니다. 그는 49.42%의 득표율로 당선되었습니다. 그는 민심이 정권 교체를 선택한 조기 대선에서 당선되었습니다.


### Simple Retriever 이용

In [ ]:
class SimpleRetriever(BaseRetriever):
    docs: list[Document]
    k: int = 5

    def _get_relevant_documents(self, query: str) -> list[Document]:
        return self.docs[:self.k]


document = Document(
    page_content='2025년 6월 3일에 당선된 제 21대 대통령은 더불어민주당 이재명이다.',
    metadata={'source': 'https://example.com'}
)

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

prompt = PromptTemplate.from_template("""
다음의 context를 읽고, 질문에 답해줘.
context: {context}
질문: {question}
""")

retriever = SimpleRetriever(docs=[document], k=1)
context = retriever.invoke('한국의 대통령은?')
# print(context)

chain = prompt | llm

response = chain.invoke({
    'question': '한국의 대통령은?',
    'context': retriever.invoke('한국의 대통령은?')
})
# print(response.content)

chain = {
    'context': retriever,
    'question': RunnablePassthrough()
} | prompt | llm
response = chain.invoke('한국의 대통령은?')
print(response.content)

한국의 대통령은 더불어민주당 이재명입니다.


### Wikipedia Retriever

In [10]:
from langchain.schema.runnable import RunnablePassthrough
from langchain_community.retrievers import WikipediaRetriever
retriever = WikipediaRetriever()

chain = {
    'context': retriever,
    'question': RunnablePassthrough()
} | prompt | llm

context = retriever.invoke('한국의 대통령은?')
print(context)

response = chain.invoke('한국의 대통령은?')
print(response.content)

[Document(metadata={'title': 'Democratic Party (South Korea, 2015)', 'summary': "The Democratic Party of Korea (DPK or DP; Korean: 더불어민주당, lit.\u2009'Together Democratic Party') is a liberal political party in South Korea. The DPK and its rival, the People Power Party (PPP), form the two major political parties of South Korea. It is the ruling party following the victory of Lee Jae Myung at the 2025 presidential election, and has been the largest party in the National Assembly since 2016, controlling a majority since 2020. It was previously the ruling party under Moon Jae-in from 2017 to 2022.\nThe Democratic Party was founded as the New Politics Alliance for Democracy (NPAD; 새정치민주연합) on 26 March 2014 as a merger between the previous Democratic Party and the preparatory committee of the New Political Vision Party (NPVP) led by Ahn Cheol-soo. The party changed its name to the current name on 28 December 2015. In 2022, the Democratic Party, the Open Democratic Party, and New Wave merged 

### Vector Store as Retriever

In [ ]:
from abc import ABC
from typing import Any
# from langchain_core.retrievers import VectorStoreRetriever

from langchain_core.callbacks import CallbackManagerForRetrieverRun


class VectorStoreRetriever(BaseRetriever):
    def __init__(self, vector_store, tags: list[str] = None, **kwargs: Any):
        super().__init__(**kwargs)
        self.vector_store = vector_store
        self.tags = tags or []

    def _get_relevant_documents(self, query, *, run_manager: CallbackManagerForRetrieverRun, **kwargs: Any) -> list[Document]:
        return self.vector_store.similarity_search(query, **kwargs)


class VectorStore(ABC):
    def as_retriever(self, **kwargs: Any) -> VectorStoreRetriever:
        raise VectorStoreRetriever(vector_store=self, **kwargs)

In [29]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
#  embeddings = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")

vector_store = InMemoryVectorStore(embeddings)

docs = [
    Document(page_content='2025년 6월 3일에 당선된 제 21대 대통령은 더불어민주당 이재명이다.',
             metadata={'source': 'https://example.com'}),
    Document(page_content='삼성 가우스는 삼성전자의 멀티모달 모델의 생성형 인공지능이다.',
             metadata={'source': 'https://example.com'}),
]
vector_store.add_documents(docs)

similar_docs = vector_store.similarity_search('한국의 대통령은?', k=2)
print(similar_docs)

prompt = PromptTemplate.from_template("""
다음의 context를 읽고, 질문에 답해줘.
context: {context}
질문: {question}
""")

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
chain = {
    'context': vector_store.as_retriever(),
    'question': RunnablePassthrough()
} | prompt | llm

response = chain.invoke('한국의 대통령은?')
print(response.content)

[Document(id='38392d7b-e05b-41c0-b474-f83c7a345a5f', metadata={'source': 'https://example.com'}, page_content='삼성 가우스는 삼성전자의 멀티모달 모델의 생성형 인공지능이다.'), Document(id='3a26a025-7da4-4e72-a405-bf457a4c7188', metadata={'source': 'https://example.com'}, page_content='2025년 6월 3일에 당선된 제 21대 대통령은 더불어민주당 이재명이다.')]
한국의 대통령은 더불어민주당 이재명입니다.


### Web Base Loader

In [36]:
from langchain_google_genai import ChatGoogleGenerativeAI
import bs4
import os
from langchain_community.document_loaders import WebBaseLoader

os.environ["USER_AGENT"] = "Mozilla/5.0"

loader = WebBaseLoader(
    web_path="https://n.news.naver.com/article/001/0015568637",
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer('article')
    )
)

docs = loader.load()
#  print(docs[0].page_content)

llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash')

response = llm.invoke(f'다음 기사를 한 문장으로 요약해줘 : {docs[0].page_content}')
print(response.content)

오픈AI의 최신 모델 GPT-5가 기대 이하의 성능으로 오류와 잘못된 답변을 연발하며 사용자들의 조롱을 받고, 이전 버전으로 되돌리는 소동까지 벌어졌다.


### PyPDFLoader

In [ ]:
# pip install pypdf
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data\\AI 에이전트 동향.pdf", mode='single')
docs = loader.load()

with open("data\\AI 에이전트 동향.txt", "w", encoding="utf-8") as f:
    f.write(docs[0].page_content)

### RecursiveCharacterTextSplitter

In [ ]:
with open('data\\AI 에이전트 동향.txt', 'r', encoding='utf-8') as f:
    file = f.read()

from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter()

docs = splitter.create_documents([file])

with open('data\\AI 에이전트 동향_splitted.txt', 'w', encoding='utf-8') as f:
    f.write(f'Number of splitted documents: {len(docs)}')
    for doc in docs:
        f.write(f'\n\n---\n\n{doc.page_content}')

### MarkdownHeaderTextSplitter

In [22]:
with open('data\\langchain.md', 'r', encoding='utf-8') as f:
    file = f.read()

from langchain_text_splitters import MarkdownHeaderTextSplitter
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[('#', 'Chapter'), ('##', 'Section')],
    strip_headers=False,
)

docs = splitter.split_text(file)
for i, doc in enumerate(docs):
    print(i, 'th\n', doc.page_content[:100])

0 th
 # Introduction  
**LangChain** is a framework for developing applications powered by large language 
1 th
 ## Architecture  
The LangChain framework consists of multiple open-source libraries. Read more in t
2 th
 ## Guides  
### [Tutorials](/docs/tutorials)  
If you're looking to build something specific or are 
3 th
 ## Ecosystem  
### [🦜🛠️ LangSmith](https://docs.smith.langchain.com)
Trace and evaluate your languag
4 th
 ## Additional resources  
### [Versions](/docs/versions/v0_3/)
See what changed in v0.3, learn how t


### SemanticChunker

In [31]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

# pip install langchain_experimental
splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type='percentile',
    breakpoint_threshold_amount=50,
)

docs = splitter.create_documents([file])
for i, doc in enumerate(docs):
    print(i, 'th\n', doc.page_content[:50])

0 th
 # Introduction

**LangChain** is a framework for d
1 th
 - **Deployment**: Turn your LangGraph applications
2 th
 See the [integrations](/docs/integrations/provider
3 th
 import ChatModelTabs from "@theme/ChatModelTabs";

4 th
 :::

## Architecture

The LangChain framework cons
5 th
 Read more in the
[Architecture](/docs/concepts/arc
6 th
 - **Integration packages** (e.g. `langchain-openai
7 th
 - **`langchain-community`**: Third-party integrati
8 th
 See [LangGraph documentation](https://langchain-ai
9 th
 ## Guides

### [Tutorials](/docs/tutorials)

If yo
10 th
 This is the best place to get started. These are t
11 th
 ### [How-to guides](/docs/how_to)

[Here](/docs/ho
12 th
 These how-to guides don’t cover topics in depth – 
13 th
 ### [Conceptual guide](/docs/concepts)

Introducti
14 th
 For a deeper dive into LangGraph concepts, check o
15 th
 If you're looking to get up and running quickly wi
16 th
 ## Ecosystem

### [🦜🛠️ LangSmith](https://docs.smi
17 th
 ### [🦜🕸️ LangGrap

### LLM Splitter

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnablePassthrough
with open('data\\AI 에이전트 동향.txt', 'r', encoding='utf-8') as f:
    file = f.read()

from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template('''
다음의 context를 읽고, 질문에 답해줘.
context: {context}
질문: {question}
''')

llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash')

# print(file)
chain = {
    'context': lambda x: file,
    'question': RunnablePassthrough(),
} | prompt | llm
response = chain.invoke('AI 에이전트 동향에 대해 5줄로 줄바꿈해서 보기쉽고 간략하게 정리해서 알려줘')
print(response.content)

AI 에이전트는 생성형 AI 확산과 함께 관심이 급증하며, 향후 시장이 빠르게 성장할 것으로 예상됩니다. 빅테크 기업들이 AI 에이전트 시장에 진출하여 다양한 수익 모델을 창출하고 있지만, 기술적, 사회적, 윤리적, 법적 문제도 내포하고 있습니다. AI 에이전트는 고객 서비스, 개인 비서, 자율 주행 차량 등 다양한 산업에 활용될 수 있으며, 개인화 서비스와 실시간 대응 등 혁신적인 서비스를 제공할 수 있습니다. Markets&Markets는 전 세계 AI 에이전트 시장이 2024년 51억 달러에서 2030년 471억 달러로 성장할 것으로 전망했습니다. Gartner는 2025년 최상위 10대 전략 기술 트렌드에서 AI 에이전트를 최우선 기술로 제시하며, 2028년까지 엔터프라이즈 소프트웨어 애플리케이션의 33%에 에이전트 AI가 포함될 것으로 예상했습니다.


### 과제
- file 정보 주고 prompt를 통해 정보 정리 요청청
- 결과를 받아서 vector db에 저장
- vector db 를 retriever로 써서 질문 답변변

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnablePassthrough
from pydantic import BaseModel, Field, ValidationError
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.exceptions import OutputParserException
import re
from typing import Any
import json


class RobustPydanticOutputParser(PydanticOutputParser):
    """
    LLM 출력 또는 Python 객체를 Pydantic 모델로 안전하게 변환하는 파서.
    - 문자열: 후처리(clean ```json) 후 parse
    - list/dict: 바로 Pydantic 객체로 변환
    - JSON 형식 강제 검사 및 ValidationError 처리
    """

    def parse(self, obj: Any):
        # 이미 list/dict이면 바로 Pydantic 객체 생성
        if isinstance(obj, (list, dict)):
            try:
                return self.pydantic_object.parse_obj(obj)
            except ValidationError as e:
                raise OutputParserException(
                    f"Validation failed on Python object: {e}\nInput: {obj}")

        # 문자열이면 후처리
        if isinstance(obj, str):
            # ```json 제거 및 공백 strip
            cleaned = re.sub(r"```json|```", "", obj).strip()
            # JSON 형식인지 강제 검사
            try:
                data = json.loads(cleaned)
            except json.JSONDecodeError as e:
                raise OutputParserException(
                    f"Invalid JSON format after cleaning: {e}\nCleaned text: {cleaned}")

            # Pydantic 객체 생성
            try:
                return self.pydantic_object.parse_obj(data)
            except ValidationError as e:
                raise OutputParserException(
                    f"Pydantic validation failed: {e}\nData: {data}")

        raise OutputParserException(f"Unsupported input type: {type(obj)}")


with open('data\\AI 에이전트 동향_short.txt', 'r', encoding='utf-8') as f:
    file = f.read()

prompt = PromptTemplate.from_template('''
다음의 context 를 읽고, 의미있는 단위로 쪼개서 반드시 format 예제와 같은 형식으로 답해주세요. 

context: {context}
format: 
[
  {{"title": "제목", "content": "내용"}},
  ...
]

모든 content는 반드시 내용이 있어야 하며, null이 되면 안됩니다.
절대로 ```json 같은 코드 블록을 사용하지 마세요.  
오직 [ ... ] JSON 배열 형식으로만 답하세요.  
다른 텍스트, 설명, 마크다운은 금지입니다.
''')


class MeaningfulChunk(BaseModel):
    title: str = Field(description="제목")
    content: str = Field(description="내용")


class MeaningfulChunkList(BaseModel):
    chunks: list[MeaningfulChunk] = Field(description="의미있는 단위로 쪼개진 콘텐츠 목록")


output_parser = RobustPydanticOutputParser(pydantic_object=MeaningfulChunkList)

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

# print(file)
chain = {
    'context': lambda x: file,
    # 'format': lambda x: output_parser.get_format_instructions()
} | prompt | llm  # | output_parser
response = chain.invoke({})

output_parser.parse(response.content)

C:\Users\usejen_id\AppData\Local\Temp\ipykernel_4008\2690495260.py:39: PydanticDeprecatedSince20: The `parse_obj` method is deprecated; use `model_validate` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  return self.pydantic_object.parse_obj(data)


OutputParserException: Pydantic validation failed: 1 validation error for MeaningfulChunkList
  Input should be a valid dictionary or instance of MeaningfulChunkList [type=model_type, input_value=[{'title': 'AI 에이전... 주도할 것이다.'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
Data: [{'title': 'AI 에이전트 동향 개요', 'content': '최근 생성형 AI의 확산과 함께 인간과 상호작용이 가능한 AI 에이전트에 대한 관심이 급증하고 있으며, 향후 몇 년 안에 관련 시장이 급속히 성장할 것으로 예상된다. 빅테크 기업들이 AI 에이전트 시장에 잇따라 진출하며 다양한 수익 모델을 창출하고 있는 상황이다. 그러나 AI 에이전트는 기술적, 사회적, 윤리적, 법적 문제를 초래할 가능성도 내포하고 있다. 앞으로 AI 에이전트는 기술 혁신과 사회 변화를 주도하는 핵심 요소로 자리 잡으며, 기업의 업무방식과 개인의 삶에 깊숙이 통합되어 획기적인 변화를 이끌어갈 것으로 전망된다.'}, {'title': 'AI 에이전트 정의 (학술적 관점)', 'content': 'AI 에이전트에 대한 합의된 학술적 정의는 부재하나, 관련 연구에서는 AI 에이전트를 단순한 AI 모델이나 알고리즘과 구별하여 설명하며, 특히 상호작용과 독립적인 의사결정 능력을 강조한다. 스튜어트 러셀(Stuart Russell)과 피터 노비그(Peter Norvig)는 2021년 저서 ‘인공지능 – 현대적 접근법’에서 에이전트(Agent)는 센서를 통해 환경을 인식하고 센서가 액추에이터를 통해 해당 환경에 작용하는 것으로, 합리적 에이전트(Rational Agent)는 최선의 결과를 달성하기 

In [44]:
print(len(parsed_data))
print(json.dumps(parsed_data, ensure_ascii=False, indent=2))

18
[
  {
    "title": "문서 정보",
    "content": "SPRi AI Brief Special | 2024년 12월호"
  },
  {
    "title": "AI 에이전트 동향",
    "content": "최근 생성형 AI의 확산과 함께 인간과 상호작용이 가능한 AI 에이전트에 대한 관심이 급증하고 있으며, 향후 몇 년 안에 관련 시장이 급속히 성장할 것으로 예상된다. 빅테크 기업들이 AI 에이전트 시장에 잇따라 진출하며 다양한 수익 모델을 창출하고 있는 상황이다. 그러나 AI 에이전트는 기술적, 사회적, 윤리적, 법적 문제를 초래할 가능성도 내포하고 있다. 앞으로 AI 에이전트는 기술 혁신과 사회 변화를 주도하는 핵심 요소로 자리 잡으며, 기업의 업무방식과 개인의 삶에 깊숙이 통합되어 획기적인 변화를 이끌어갈 것으로 전망된다."
  },
  {
    "title": "1. AI 에이전트(AI Agent)의 도입 및 부상",
    "content": ""
  },
  {
    "title": "1) AI 에이전트의 정의",
    "content": "AI 에이전트에 대한 합의된 학술적 정의는 부재하나, 관련 연구에서는 AI 에이전트를 단순한 AI 모델이나 알고리즘과 구별하여 설명하며, 특히 상호작용과 독립적인 의사결정 능력을 강조"
  },
  {
    "title": "AI 에이전트 정의 - 스튜어트 러셀 & 피터 노비그",
    "content": "스튜어트 러셀(Stuart Russell)과 피터 노비그(Peter Norvig)는 2021년 출판한 저서 ‘인공지능 – 현대적 접근법’에서 에이전트(Agent)는 센서를 통해 환경을 인식하고 센서가 액추에이터를 통해 해당 환경에 작용하는 것으로, 합리적 에이전트(Rational Agent)는 최선의 결과를 달성하기 위해 행동하거나, 불확실성이 있는 경우 최선의 기대 결과를 얻기 위해 행동하는 행위자로 정의"
  },
  {
    "title": "AI 에이전트 

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

vector_store = InMemoryVectorStore(embeddings)

for data in parsed_data:
    vector_store.add_documents(
        [Document(page_content=data['content'], metadata={'title': data['title']})])

vector_store

ValidationError: 1 validation error for Document
page_content
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type

In [53]:
prompt = PromptTemplate.from_template("""
다음의 context를 읽고, 질문에 답해줘.
답변할 때는 줄바꿈을 적절히 해서 보기 쉽게 해줘.
context: {context}
질문: {question}
""")

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
chain = {
    'context': vector_store.as_retriever(),
    'question': RunnablePassthrough()
} | prompt | llm

response = chain.invoke('ISO/IEC에서 정의한 AI 에이전트는?')
print(response.content)

제공된 문서에는 ISO/IEC에서 정의한 AI 에이전트에 대한 정보가 없습니다.


In [30]:
prompt

PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n다음의 context 를 읽고, 의미있는 단위로 쪼개서 반드시 format의 JSON 형식으로 답해줘\ncontext: {context}\nformat : \n[\n  {{"title": "제목", "content": "내용"}},\n  ...\n]\n')

In [20]:
output_parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"title": {"description": "제목", "title": "Title", "type": "string"}, "content": {"description": "내용", "title": "Content", "type": "string"}}, "required": ["title", "content"]}\n```'

In [34]:
response.content

'```json\n[\n  {"title": "문서 정보", "content": "SPRi AI Brief Special | 2024년 12월호"},\n  {"title": "보고서 제목", "content": "AI 에이전트 동향 – 빅테크 기업의 AI 에이전트 사례를 중심으로"},\n  {"title": "서론", "content": "최근 생성형 AI의 확산과 함께 인간과 상호작용이 가능한 AI 에이전트에 대한 관심이 급증하고 있으며, 향후 몇 년 안에 관련 시장이 급속히 성장할 것으로 예상된다. 빅테크 기업들이 AI 에이전트 시장에 잇따라 진출하며 다양한 수익 모델을 창출하고 있는 상황이다. 그러나 AI 에이전트는 기술적, 사회적, 윤리적, 법적 문제를 초래할 가능성도 내포하고 있다. 앞으로 AI 에이전트는 기술 혁신과 사회 변화를 주도하는 핵심 요소로 자리 잡으며, 기업의 업무방식과 개인의 삶에 깊숙이 통합되어 획기적인 변화를 이끌어갈 것으로 전망된다."},\n  {"title": "1. AI 에이전트(AI Agent)의 도입 및 부상", "content": ""},\n  {"title": "1) AI 에이전트의 정의", "content": "AI 에이전트에 대한 합의된 학술적 정의는 부재하나, 관련 연구에서는 AI 에이전트를 단순한 AI 모델이나 알고리즘과 구별하여 설명하며, 특히 상호작용과 독립적인 의사결정 능력을 강조"},\n  {"title": "스튜어트 러셀과 피터 노비그의 정의", "content": "에이전트(Agent)는 센서를 통해 환경을 인식하고 센서가 액추에이터를 통해 해당 환경에 작용하는 것으로, 합리적 에이전트(Rational Agent)는 최선의 결과를 달성하기 위해 행동하거나, 불확실성이 있는 경우 최선의 기대 결과를 얻기 위해 행동하는 행위자로 정의"},\n  {"title": "알란 찬 등의 정의", "content": "많은 AI 개발자들이 더 큰 자율성, 외부 도구나 서비스 접근, 장기적 목표 달성을 위해 안정적으로 

### Embedding

In [ ]:
# pip install sentence-transformers
from sklearn.metrics.pairwise import cosine_similarity
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_google_genai import GoogleGenerativeAIEmbeddings
# embeddings=GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
embeddings = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")
docs = [
    "한국의 수도는 서울입니다.",
    "중국의 수도는 북경입니다.",
    "삼성전자의 본사는 수원에 있습니다.",
    "파이썬은 프로그래밍 언어입니다.",
]
query = "한국의 수도는?"
docs = [
    "사과는 빨간색도 있고, 초록색도 있습니다. 나는 빨간 사과는 좋아하지만 초록 사과는 싫어합니다.",
    "바나나는 노란색입니다. 바나나는 맛있습니다. 나는 바나나를 좋아합니다.",
    "포도는 보라색입니다. 포도는 작고 달콤합니다. 포도는 씨가 있어서 먹기 불편합니다. 나는 포도를 싫어합니다.",
    "나는 낚시를 좋아합니다. 낚시는 물고기를 잡는 재미가 있습니다. 낚시는 자연과 함께하는 활동입니다.",
    "나는 여행을 좋아합니다. 여행을 통해 다양한 과일을 맛볼 수 있습니다.",
]
query = "나는 어떤 과일을 좋아할까요?"
embedded_docs = embeddings.embed_documents(docs)
embedded_query = embeddings.embed_query(query)

similarity = cosine_similarity([embedded_query], embedded_docs)
print(similarity)

[[0.55698943 0.5663046  0.40962162 0.36441505 0.60168509]]


### Bi-Encoder vs. Cross-Encoder

In [ ]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
docs = [
    "사과는 빨간색도 있고, 초록색도 있습니다. 나는 빨간 사과는 좋아하지만 초록 사과는 싫어합니다.",
    "바나나는 노란색입니다. 바나나는 맛있습니다. 나는 바나나를 좋아합니다.",
    "포도는 보라색입니다. 포도는 작고 달콤합니다. 포도는 씨가 있어서 먹기 불편합니다. 나는 포도를 싫어합니다.",
    "나는 낚시를 좋아합니다. 낚시는 물고기를 잡는 재미가 있습니다. 낚시는 자연과 함께하는 활동입니다.",
    "나는 여행을 좋아합니다. 여행을 통해 다양한 과일을 맛볼 수 있습니다.",
]
query = "나는 어떤 과일을 좋아할까요?"
cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
scores = cross_encoder.score([(query, doc) for doc in docs])
print(scores)

c:\yul2ya\llmabok\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\usejen_id\.cache\huggingface\hub\models--BAAI--bge-reranker-v2-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


[1.8223611e-01 2.8320396e-01 3.9509014e-04 1.7941722e-05 6.6267192e-02]


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
embeddings = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")
vector_store = InMemoryVectorStore(embeddings)
docs = [
    "사과는 빨간색도 있고, 초록색도 있습니다. 나는 빨간 사과는 좋아하지만 초록 사과는 싫어합니다.",
    "바나나는 노란색입니다. 바나나는 맛있습니다. 나는 바나나를 좋아합니다.",
    "포도는 보라색입니다. 포도는 작고 달콤합니다. 포도는 씨가 있어서 먹기 불편합니다. 나는 포도를 싫어합니다.",
    "나는 낚시를 좋아합니다. 낚시는 물고기를 잡는 재미가 있습니다. 낚시는 자연과 함께하는 활동입니다.",
    "나는 여행을 좋아합니다. 여행을 통해 다양한 과일을 맛볼 수 있습니다.",
]
query = "나는 어떤 과일을 좋아할까요?"

vector_store.add_texts(docs)
response = vector_store.similarity_search(query, k=3)
print(response)

[Document(id='90e9a254-341f-450f-aa6a-a4e49f052bbd', metadata={}, page_content='나는 여행을 좋아합니다. 여행을 통해 다양한 과일을 맛볼 수 있습니다.'), Document(id='d2eeae01-e3b4-4d2a-b7d7-9afdbb52a9d0', metadata={}, page_content='바나나는 노란색입니다. 바나나는 맛있습니다. 나는 바나나를 좋아합니다.'), Document(id='309f36a4-1e3f-4f55-809d-0e43f788ecea', metadata={}, page_content='사과는 빨간색도 있고, 초록색도 있습니다. 나는 빨간 사과는 좋아하지만 초록 사과는 싫어합니다.')]


### Contextual Compression Retriever

In [76]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
reranker = CrossEncoderReranker(model=cross_encoder)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=vector_store.as_retriever()
)

prompt = PromptTemplate.from_template("""
다음의 context를 읽고, 질문에 답해줘.
context: {context}
질문: {question}
""")

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
chain = {
    'context': vector_store.as_retriever(),
    'question': RunnablePassthrough()
} | prompt | llm

response = chain.invoke('내가 좋아하는 과일은?')
print(response.content)

나는 바나나와 빨간 사과를 좋아합니다.


### Vector Store - Chroma

In [5]:
from langchain_chroma import Chroma
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name='Qwen/Qwen3-Embedding-0.6B')

# pip install langchain-chroma
vector_store = Chroma.from_texts(
    docs, embeddings, persist_directory="chroma_db"
)
retriever = vector_store.as_retriever()
searched = retriever.invoke("내가 좋아하는 과일은?")
print([s.page_content for s in searched])

['바나나는 노란색입니다. 바나나는 맛있습니다. 나는 바나나를 좋아합니다.', '바나나는 노란색입니다. 바나나는 맛있습니다. 나는 바나나를 좋아합니다.', '바나나는 노란색입니다. 바나나는 맛있습니다. 나는 바나나를 좋아합니다.', '바나나는 노란색입니다. 바나나는 맛있습니다. 나는 바나나를 좋아합니다.']


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="chroma_db"
)
cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
reranker = CrossEncoderReranker(model=cross_encoder)
retriever = ContextualCompressionRetriever(
    base_compressor=reranker, base_retriever=vector_store.as_retriever()
)
prompt = PromptTemplate.from_template("""
다음의 context를 읽고, 질문에 답해줘.
context: {context}
질문: {question}
""")

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
chain = {
    'context': retriever,
    'question': RunnablePassthrough()
} | prompt | llm

response = chain.invoke('내가 좋아하는 과일은?')
print(response.content)

나는 바나나를 좋아합니다.


### CSV Loader

In [2]:
from langchain_community.document_loaders.csv_loader import CSVLoader

loader = CSVLoader(file_path='data/restaurant_reviews.csv', encoding='utf-8')

docs = loader.load()

print(f'문서 수 : {len(docs)}')
print(docs[0])

문서 수 : 100
page_content='ID: REST_001
Taste: 60
Ambiance: 60
Service: 65
Worth_the_price: no
Menu_variety: 61
Hygienic: 65
Vegan_options: yes
Smoking_area: yes
Parking: yes
Pet_friendly: no' metadata={'source': 'data/restaurant_reviews.csv', 'row': 0}


### Generate Text Description

In [3]:
import pandas as pd
df = pd.read_csv('data/restaurant_reviews.csv')
df

,ID,Taste,Ambiance,Service,Worth_the_price,Menu_variety,Hygienic,Vegan_options,Smoking_area,Parking,Pet_friendly
0,REST_001,60,60,65,no,61,65,yes,yes,yes,no
1,REST_002,75,71,69,no,78,78,no,yes,no,no
2,REST_003,80,85,79,no,85,85,no,yes,yes,yes
3,REST_004,90,20,90,no,20,50,yes,no,yes,yes
4,REST_005,98,98,98,no,98,98,no,no,no,yes
...,...,...,...,...,...,...,...,...,...,...,...
95,REST_096,22,23,25,no,23,24,yes,yes,yes,no
96,REST_097,21,36,36,no,85,80,yes,no,no,yes
97,REST_098,20,26,31,no,20,90,yes,no,no,no
98,REST_099,65,61,65,no,86,86,yes,yes,no,no


In [7]:
# pip install pandasdf = pd.read_csv('data/restaurant_reviews.csv')

from langchain_google_genai import ChatGoogleGenerativeAI
import pandas as pd
df = pd.read_csv('data/restaurant_reviews.csv')
llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash')


def score_to_text(score):
    score = int(score)
    if score >= 90:
        return "매우 좋음"
    elif score >= 80:
        return "좋음"
    elif score >= 60:
        return "보통"
    else:
        return "비추"


def generate_description(row):
    desc = f"{row.ID}는 맛 점수 {score_to_text(row.Taste)}, 분위기 {score_to_text(row.Ambiance)}, 서비스 {score_to_text(row.Service)}로 평가됩니다. "
    desc += f"메뉴 다양성은 {score_to_text(row.Menu_variety)}, 위생 점수는 {score_to_text(row.Hygienic)}입니다. "
    desc += f"가격 대비 가치 점수는 {'좋음' if row.Worth_the_price != 'no' else '안좋음'},"
    desc += f"비건옵션: {'있음' if row.Vegan_options == 'yes' else '없음'}, "
    desc += f"흡연구역: {'있음' if row.Smoking_area == 'yes' else '없음'}, "
    desc += f"주차공간: {'있음' if row.Parking == 'yes' else '없음'}, "
    desc += f"반려동물: {'가능' if row.Pet_friendly == 'yes' else '불가'}."
    return desc


df['Description'] = df.apply(generate_description, axis=1)

df.to_csv('data/restaurant_reviews_with_descriptions.csv', index=False)
df

,ID,Taste,Ambiance,Service,Worth_the_price,Menu_variety,Hygienic,Vegan_options,Smoking_area,Parking,Pet_friendly,Description
0,REST_001,60,60,65,no,61,65,yes,yes,yes,no,"REST_001는 맛 점수 보통, 분위기 보통, 서비스 보통로 평가됩니다. 메뉴 다..."
1,REST_002,75,71,69,no,78,78,no,yes,no,no,"REST_002는 맛 점수 보통, 분위기 보통, 서비스 보통로 평가됩니다. 메뉴 다..."
2,REST_003,80,85,79,no,85,85,no,yes,yes,yes,"REST_003는 맛 점수 좋음, 분위기 좋음, 서비스 보통로 평가됩니다. 메뉴 다..."
3,REST_004,90,20,90,no,20,50,yes,no,yes,yes,"REST_004는 맛 점수 매우 좋음, 분위기 비추, 서비스 매우 좋음로 평가됩니다..."
4,REST_005,98,98,98,no,98,98,no,no,no,yes,"REST_005는 맛 점수 매우 좋음, 분위기 매우 좋음, 서비스 매우 좋음로 평가..."
...,...,...,...,...,...,...,...,...,...,...,...,...
95,REST_096,22,23,25,no,23,24,yes,yes,yes,no,"REST_096는 맛 점수 비추, 분위기 비추, 서비스 비추로 평가됩니다. 메뉴 다..."
96,REST_097,21,36,36,no,85,80,yes,no,no,yes,"REST_097는 맛 점수 비추, 분위기 비추, 서비스 비추로 평가됩니다. 메뉴 다..."
97,REST_098,20,26,31,no,20,90,yes,no,no,no,"REST_098는 맛 점수 비추, 분위기 비추, 서비스 비추로 평가됩니다. 메뉴 다..."
98,REST_099,65,61,65,no,86,86,yes,yes,no,no,"REST_099는 맛 점수 보통, 분위기 보통, 서비스 보통로 평가됩니다. 메뉴 다..."


### Store to Vector Store (Chroma)

In [ ]:
from langchain.schema.runnable import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
import pandas as pd
df = pd.read_csv('data/restaurant_reviews_with_descriptions.csv')

docs = [
    Document(
        page_content=row['Description'],
        metadata={
            key: row[key]
            for key in row.index if key != 'Description'
        }
    )
    for _, row in df.iterrows()
]

# print(docs[0])

embedding = HuggingFaceEmbeddings(model_name='Qwen/Qwen3-Embedding-0.6B')

db = Chroma.from_documents(
    docs, embedding, persist_directory='chroma_restaurant_2'
)

searched = db.similarity_search('주차공간과 흡연공간이 있는 맛점수가 높은 레스토랑은?', k=3)
print([s.page_content for s in searched])

prompt = PromptTemplate.from_template(
    "다음 내용을 참고해서 질문에 답하세요.\n\n참고할 내용:{context}\n\n질문: {question}"
)

llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash')

chain = {
    'context': db.as_retriever(),
    'question': RunnablePassthrough()
} | prompt | llm

response = chain.invoke('주차공간과 흡연공간이 있는 맛점수가 높은 레스토랑은?')
print(response.content)

['REST_005는 맛 점수 매우 좋음, 분위기 매우 좋음, 서비스 매우 좋음로 평가됩니다. 메뉴 다양성은 매우 좋음, 위생 점수는 매우 좋음입니다. 가격 대비 가치 점수는 안좋음,비건옵션: 없음, 흡연구역: 없음, 주차공간: 없음, 반려동물: 가능.', 'REST_071는 맛 점수 매우 좋음, 분위기 매우 좋음, 서비스 매우 좋음로 평가됩니다. 메뉴 다양성은 매우 좋음, 위생 점수는 매우 좋음입니다. 가격 대비 가치 점수는 안좋음,비건옵션: 있음, 흡연구역: 있음, 주차공간: 없음, 반려동물: 불가.', 'REST_073는 맛 점수 매우 좋음, 분위기 매우 좋음, 서비스 매우 좋음로 평가됩니다. 메뉴 다양성은 매우 좋음, 위생 점수는 매우 좋음입니다. 가격 대비 가치 점수는 안좋음,비건옵션: 있음, 흡연구역: 있음, 주차공간: 없음, 반려동물: 불가.']
주어진 정보에서 주차 공간은 없고, 흡연 공간이 있는 레스토랑은 REST_071과 REST_073입니다. 맛 점수는 두 곳 모두 매우 좋습니다.


In [9]:
from langchain_chroma import Chroma
db = Chroma(
    embedding_function=embedding,
    persist_directory='chroma_restaurant_2'
)

searched = db.similarity_search(
    "흡연 구역과 주차장이 있고 맛 평가가 80점 이상인 식당을 추천해줘.", k=3
)
print(searched)

searched = db.similarity_search(
    "흡연 구역과 주차장이 있는 식당을 알려줘.", k=3,
    filter={"Taste": {"$gte": 70}}
)
for s in searched:
    print(s.metadata['ID'], '흡연:', s.metadata['Smoking_area'],
          ' 주차:', s.metadata['Parking'], ' 맛:', s.metadata['Taste'])

[Document(id='844e41cb-af3f-4ed8-806f-100c0b057ef4', metadata={'Hygienic': 36, 'Vegan_options': 'yes', 'Worth_the_price': 'yes', 'Ambiance': 20, 'Menu_variety': 31, 'ID': 'REST_084', 'Pet_friendly': 'yes', 'Service': 20, 'Smoking_area': 'yes', 'Taste': 90, 'Parking': 'yes'}, page_content='REST_084는 맛 점수 매우 좋음, 분위기 비추, 서비스 비추로 평가됩니다. 메뉴 다양성은 비추, 위생 점수는 비추입니다. 가격 대비 가치 점수는 좋음,비건옵션: 있음, 흡연구역: 있음, 주차공간: 있음, 반려동물: 가능.'), Document(id='8fac63b7-436a-4ca5-9cba-63cbf6f0f095', metadata={'Hygienic': 5, 'Pet_friendly': 'yes', 'Menu_variety': 5, 'Taste': 1, 'Vegan_options': 'no', 'Ambiance': 5, 'Parking': 'yes', 'Service': 5, 'Worth_the_price': 'no', 'ID': 'REST_080', 'Smoking_area': 'yes'}, page_content='REST_080는 맛 점수 비추, 분위기 비추, 서비스 비추로 평가됩니다. 메뉴 다양성은 비추, 위생 점수는 비추입니다. 가격 대비 가치 점수는 안좋음,비건옵션: 없음, 흡연구역: 있음, 주차공간: 있음, 반려동물: 가능.'), Document(id='bfab9bc4-62ac-4b09-90fe-a4db5f8e280c', metadata={'Menu_variety': 36, 'Worth_the_price': 'no', 'Parking': 'no', 'Ambiance': 86, 'Smoking_area': 'no', 'ID': 'R

In [11]:
from langchain_chroma import Chroma
db = Chroma(
    embedding_function=embedding, persist_directory="chroma_db_restaurant",
    collection_name="restaurant_reviews"
)
searched = db.get(
    where={"$and": [
        {"Smoking_area": "yes"},
        {"Parking": "yes"},
        {"Taste": {"$gte": 80}}
    ]
    },
)
print(searched)
for m in searched["metadatas"]:
    print(m)

{'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': []}


### Self query retriever

In [13]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.base import AttributeInfo

metadata_field_info = [
    AttributeInfo(
        name="Taste",
        description="the taste of the food provided by the restaurant. Rated between 1 to 100",
        type="integer",
    ),
    AttributeInfo(
        name="Ambiance",
        description="the ambiance of the restaurant. Rated between 1 to 100",
        type="integer",
    ),
    AttributeInfo(
        name="Menu_variety",
        description="the Menu_variety of the restaurant. Rated between 1 to 100",
        type="integer",
    ),
    AttributeInfo(
        name="Hygienic",
        description="the Hygienic of the restaurant. Rated between 1 to 100",
        type="integer",
    ),
    AttributeInfo(
        name="Service",
        description="the service of the restaurant. Rated between 1 to 100",
        type="integer",
    ),
    AttributeInfo(
        name="Worth_the_price",
        description="the worth the price of the restaurant. no or yes",
        type="String",
    ),
    AttributeInfo(
        name="Vegan_options",
        description="have the vegan options. no or yes",
        type="String",
    ),
    AttributeInfo(
        name="Smoking_area",
        description="have the smocking area. no or yes",
        type="String",
    ),
    AttributeInfo(
        name="Parking",
        description="have the parking area. no or yes",
        type="String",
    ),
    AttributeInfo(
        name="Pet_friendly",
        description="the pet friendly of the restaurant. no or yes",
        type="String",
    ),
]


llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash')
embeddings = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")

vector_store = Chroma(embedding_function=embeddings,
                      persist_directory="chroma_restaurant_2")
retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vector_store,
    document_contents="Restaurant review data",
    metadata_field_info=metadata_field_info,
)

response = retriever.invoke('주차장이 있고 맛 평가가 80점 이상인 식당은?')
for doc in response:
    print(doc.metadata)

{'Worth_the_price': 'no', 'Smoking_area': 'yes', 'Taste': 86, 'Ambiance': 86, 'ID': 'REST_055', 'Service': 86, 'Pet_friendly': 'yes', 'Parking': 'yes', 'Menu_variety': 85, 'Hygienic': 85, 'Vegan_options': 'no'}
{'Worth_the_price': 'no', 'Smoking_area': 'no', 'ID': 'REST_004', 'Hygienic': 50, 'Ambiance': 20, 'Taste': 90, 'Vegan_options': 'yes', 'Pet_friendly': 'yes', 'Menu_variety': 20, 'Parking': 'yes', 'Service': 90}
{'Parking': 'yes', 'Smoking_area': 'no', 'Vegan_options': 'yes', 'Taste': 86, 'Ambiance': 56, 'Service': 54, 'Worth_the_price': 'no', 'Pet_friendly': 'no', 'Menu_variety': 86, 'Hygienic': 86, 'ID': 'REST_042'}
{'Parking': 'yes', 'Vegan_options': 'yes', 'Worth_the_price': 'yes', 'Ambiance': 20, 'Service': 20, 'ID': 'REST_084', 'Pet_friendly': 'yes', 'Taste': 90, 'Menu_variety': 31, 'Hygienic': 36, 'Smoking_area': 'yes'}


In [15]:
from langchain.chains.query_constructor.base import get_query_constructor_prompt
prompt = get_query_constructor_prompt(
    "Restaurant review data",  # 문서 내용 설명
    metadata_field_info,  # 메타데이터 필드 정보
)
prompt.pretty_print()

Your goal is to structure the user's query to match the request schema provided below.

<< Structured Request Schema >>
When responding use a markdown code snippet with a JSON object formatted in the following schema:

```json
{
    "query": string \ text string to compare to document contents
    "filter": string \ logical condition statement for filtering documents
}
```

The query string should contain only text that is expected to match the contents of documents. Any conditions in the filter should not be mentioned in the query as well.

A logical condition statement is composed of one or more comparison and logical operation statements.

A comparison statement takes the form: `comp(attr, val)`:
- `comp` (eq | ne | gt | gte | lt | lte | contain | like | in | nin): comparator
- `attr` (string):  name of attribute to apply the comparison to
- `val` (string): is the comparison value

A logical operation statement takes the form `op(statement1, statement2, ...)`:
- `op` (and | or | not

In [ ]:
from langchain.chains.query_constructor.base import StructuredQueryOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
output_parser = StructuredQueryOutputParser.from_components()
query_constructor = prompt | llm | output_parser
query_output = query_constructor.invoke("주차장이 있고 맛 평가가 80점 이상인 식당은?")
print(query_output)

query=' ' filter=Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='Parking', value='yes'), Comparison(comparator=<Comparator.GTE: 'gte'>, attribute='Taste', value='80')]) limit=None


In [17]:
from langchain_community.query_constructors.chroma import ChromaTranslator
translator = ChromaTranslator()
filter = translator.visit_operation(query_output.filter)
print(filter)

{'$and': [{'Parking': {'$eq': 'yes'}}, {'Taste': {'$gte': '80'}}]}


In [19]:
from langchain_community.utilities import SQLDatabase
from sqlalchemy import create_engine
import pandas as pd     # Importing pandas for data manipulation
df = pd.read_csv('data/restaurant_reviews.csv')

engine = create_engine("sqlite:///restaurant.db")

df.to_sql("restaurant", engine, if_exists="replace", index=False)

db = SQLDatabase(engine=engine)

print(db.dialect)                   # sqlite
print(db.get_usable_table_names())  # ['restaurant']
print(db.get_table_info())

searched = db.run(
    "SELECT * FROM restaurant WHERE Smoking_Area='yes' and Parking='yes' and Taste >= 80;")
print(eval(searched))       # searched가 문자열이다.
for row in eval(searched):
    print(row[0])

sqlite
['restaurant']

CREATE TABLE restaurant (
	"ID" TEXT, 
	"Taste" BIGINT, 
	"Ambiance" BIGINT, 
	"Service" BIGINT, 
	"Worth_the_price" TEXT, 
	"Menu_variety" BIGINT, 
	"Hygienic" BIGINT, 
	"Vegan_options" TEXT, 
	"Smoking_area" TEXT, 
	"Parking" TEXT, 
	"Pet_friendly" TEXT
)

/*
3 rows from restaurant table:
ID	Taste	Ambiance	Service	Worth_the_price	Menu_variety	Hygienic	Vegan_options	Smoking_area	Parking	Pet_friendly
REST_001	60	60	65	no	61	65	yes	yes	yes	no
REST_002	75	71	69	no	78	78	no	yes	no	no
REST_003	80	85	79	no	85	85	no	yes	yes	yes
*/
[('REST_003', 80, 85, 79, 'no', 85, 85, 'no', 'yes', 'yes', 'yes'), ('REST_010', 80, 80, 80, 'yes', 80, 80, 'no', 'yes', 'yes', 'no'), ('REST_012', 80, 99, 74, 'yes', 80, 80, 'yes', 'yes', 'yes', 'yes'), ('REST_055', 86, 86, 86, 'no', 85, 85, 'no', 'yes', 'yes', 'yes'), ('REST_084', 90, 20, 20, 'yes', 31, 36, 'yes', 'yes', 'yes', 'yes')]
REST_003
REST_010
REST_012
REST_055
REST_084


In [21]:
from langchain import hub
prompt = hub.pull("rlm/text-to-sql")
prompt = prompt.partial(table_info=db.get_table_info(), dialect=db.dialect,
                        few_shot_examples="")
print(prompt)
chain = prompt | llm
response = chain.invoke("주차장이 있고 맛 평가가 높은 식당을 3개 찾아줘.")
print(response.content)

input_variables=['input'] input_types={} partial_variables={'table_info': '\nCREATE TABLE restaurant (\n\t"ID" TEXT, \n\t"Taste" BIGINT, \n\t"Ambiance" BIGINT, \n\t"Service" BIGINT, \n\t"Worth_the_price" TEXT, \n\t"Menu_variety" BIGINT, \n\t"Hygienic" BIGINT, \n\t"Vegan_options" TEXT, \n\t"Smoking_area" TEXT, \n\t"Parking" TEXT, \n\t"Pet_friendly" TEXT\n)\n\n/*\n3 rows from restaurant table:\nID\tTaste\tAmbiance\tService\tWorth_the_price\tMenu_variety\tHygienic\tVegan_options\tSmoking_area\tParking\tPet_friendly\nREST_001\t60\t60\t65\tno\t61\t65\tyes\tyes\tyes\tno\nREST_002\t75\t71\t69\tno\t78\t78\tno\tyes\tno\tno\nREST_003\t80\t85\t79\tno\t85\t85\tno\tyes\tyes\tyes\n*/', 'dialect': 'sqlite', 'few_shot_examples': ''} metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'text-to-sql', 'lc_hub_commit_hash': '794179d844347574a2e6012924720bf4beebe35d8229fa632b693807f069d612'} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['dialect', 'few_shot_examples', 'input', 'ta

## Evaluation

- LLM as Judge
- Lang Smith 
  - Observability 
    - Trace
  - Prompt Engineering
  - Evals

### Generation using RAGAS
- 데이터셋을 만들어 주는 도구

In [28]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

# 저장된 벡터 스토어 파일 경로
path = 'data/ai_trends_vector_store'

# 벡터 스토어 로드
vector_store = InMemoryVectorStore.load(path, embedding=embeddings)
vector_store

In [ ]:
# import dotenv
# dotenv.load_dotenv()

# from langchain_google_genai import ChatGoogleGenerativeAI
# llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

# from langchain_huggingface import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")

# # pip install ragas rapidfuzz
# from ragas.testset import TestsetGenerator
# generator = TestsetGenerator.from_langchain(llm, embeddings)

# docs = [Document(page_content=text) for text in texts]

# dataset=generator.generate_with_langchain_docs(docs, testset_size=10)

# df=dataset.to_pandas()
# df.to_csv('ragas/dataset.csv', index=False)

In [ ]:
from ragas.testset import TestsetGenerator
generator=TestsetGenerator(llm=llm_wrapper, embedding_model=embeddings_wrapper)
dataset=generator.generate_with_langchain_docs(docs, testset_size=10)
df=dataset.to_pandas()
df.to_csv('dataset.csv', index=False)
https://docs.ragas.io/en/stable/getstarted/rag_testset_generation

### Examples 등록
평가할 수 있는 예제들 

In [29]:
import pandas as pd
from langsmith import Client
from dotenv import load_dotenv
load_dotenv()
client = Client()

dataset = client.create_dataset(
    dataset_name="AGENT_DATASET",
    description="dataset for AI trends"
)

df = pd.read_csv("data/dataset.csv")

examples = [
    {
        "inputs": {"question": row["user_input"]},
        "outputs": {"answer": row["reference"]}
    }
    for _, row in df.iterrows()
]

client.create_examples(dataset_id=dataset.id, examples=examples)

{'example_ids': ['6c9bafc4-7d47-471a-b82f-4c0b3c7bd443',
  '80333e19-8ea1-4a60-a464-09d4875ed4d8',
  '92d056da-30e5-43a6-b8e7-08dba889de0f',
  'e568ff64-b2a8-4700-b718-9bf0bc01ef07',
  '11fc11fd-0750-4ddd-a92b-14b1f1b93394',
  '1506ec80-83be-45e7-95f9-e4c68eb445c8',
  'e3c55591-f60f-44b0-b8d7-718090678500',
  '5af9bc4a-216c-40a3-acff-99a8a63ed87a',
  'a3ce552d-3d56-4561-8954-71a866785fd8',
  'aa7e73ba-4309-48bf-a69d-73917a76e11f',
  'dfa80e9e-6557-4ba9-a04b-dc32162d9811',
  '409faf17-2413-4bff-ab8b-9c2e84b06aed'],
 'count': 12}

In [ ]:
from langsmith.evaluation import evaluate, LangChainStringEvaluator
from langsmith.schemas import Run, Example
from langchain_core.runnables import RunnableLambda
chain = RunnableLambda(lambda x: x)


def my_score_evaluator(run: Run, example: Example) -> dict:
    #   (run.outputs["result"], example.outputs["answer"])로부터 score를 계산
    import random
    return {"key": "my_score", "score": random.randint(1, 11)}   # 랜덤 평가


experiment_results = evaluate(
    lambda inputs: {"result": chain.invoke(inputs["question"])},
    data="AGENT_DATASET",
    evaluators=[my_score_evaluator],
    experiment_prefix="RAG_EVAL",
)

View the evaluation results for experiment: 'RAG_EVAL-5f9f89c6' at:
https://smith.langchain.com/o/3ea1fbac-6bd8-4944-8e26-03912fd4d074/datasets/cd259c5e-5a56-46e9-ac87-97cff47dbae2/compare?selectedSessions=a602001e-d0b0-43d3-8fe5-3d011080a0b1




0it [00:00, ?it/s]

In [38]:
from langsmith.evaluation import evaluate, LangChainStringEvaluator
from langsmith.schemas import Run, Example
from langchain_core.runnables import RunnableLambda
from langchain_google_genai import ChatGoogleGenerativeAI


qa_evalulator = LangChainStringEvaluator("qa", config={"llm": llm})
qa_evalulator.evaluator.prompt.pretty_print()

embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

# 저장된 벡터 스토어 파일 경로
path = 'data/ai_trends_vector_store'

# 벡터 스토어 로드
vector_store = InMemoryVectorStore.load(path, embedding=embeddings)

prompt = PromptTemplate.from_template(
    "다음 내용을 참고해서 질문에 답하세요.\n\n참고할 내용:{context}\n\n질문: {question}"
)

llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash')

chain = {
    'context': vector_store.as_retriever(),
    'question': RunnablePassthrough()
} | prompt | llm

# response = chain.invoke('ai 트렌드에 대해 알려줘')
# response.content
experiment_results = evaluate(
    lambda inputs: {"result": chain.invoke(inputs["question"])},
    data="AGENT_DATASET",
    evaluators=[qa_evalulator],
    experiment_prefix="RAG_EVAL",
)

You are a teacher grading a quiz.
You are given a question, the student's answer, and the true answer, and are asked to score the student answer as either CORRECT or INCORRECT.

Example Format:
QUESTION: question here
STUDENT ANSWER: student's answer here
TRUE ANSWER: true answer here
GRADE: CORRECT or INCORRECT here

Grade the student answers based ONLY on their factual accuracy. Ignore differences in punctuation and phrasing between the student answer and true answer. It is OK if the student answer contains more information than the true answer, as long as it does not contain any conflicting statements. Begin!

QUESTION: {query}
STUDENT ANSWER: {result}
TRUE ANSWER: {answer}
GRADE:
View the evaluation results for experiment: 'RAG_EVAL-9c708054' at:
https://smith.langchain.com/o/3ea1fbac-6bd8-4944-8e26-03912fd4d074/datasets/cd259c5e-5a56-46e9-ac87-97cff47dbae2/compare?selectedSessions=e817ed55-6fa9-4d56-872c-42ff9ee4ec7c




0it [00:00, ?it/s]

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 23
}
].
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing

In [ ]:
# 간결성 평가
conciseness_evalulator = LangChainStringEvaluator(
    "criteria",
    config={"criteria": "conciseness", "llm": llm}
)
# 관련성 평가
relevance_evalulator = LangChainStringEvaluator(
    "criteria",
    config={"criteria": "relevance", "llm": llm}
)
# 정확성 평가
correctness_evalulator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": "correctness", "llm": llm},
    prepare_data=lambda run, example: {
        "prediction": run.outputs["result"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

In [ ]:
# answer와 expected answer 의 similarity 반환해서 평가
embedding_evaluator = LangChainStringEvaluator(
    "embedding_distance",
    config={
        "embeddings": embeddings,
        "distance_metric": "cosine",
    },
)

In [ ]:
experiment_results = evaluate(
    lambda inputs: {"result": chain.invoke(inputs["question"])},
    data="AGENT_DATASET",
    evaluators=[qa_evalulator, conciseness_evalulator, relevance_evalulator, correctness_evalulator, embedding_evaluator],
    experiment_prefix="RAG_EVAL",
)